# Speech Emotion Detection Model Training
## Train a Custom CNN-LSTM Model for Mental Health Chatbot

This notebook trains a speech emotion recognition model using:
- **Datasets**: RAVDESS, CREMA-D, TESS, SAVEE
- **Architecture**: CNN-LSTM with attention
- **Features**: MFCC, Mel-Spectrogram, Chroma, ZCR
- **Emotions**: angry, sad, happy, fear, disgust, neutral, calm, surprised

### Requirements:
- Google Colab (GPU recommended)
- Kaggle API credentials (for dataset download)

## Step 1: Setup Environment

In [ ]:
# Install required packages
# Pin numpy<2 FIRST so everything compiles against the same ABI,
# then let pip resolve compatible versions of the rest.
!pip install -q "numpy<2.0" librosa soundfile scikit-learn tensorflow kaggle tqdm matplotlib seaborn

print("\n" + "="*60)
print("IMPORTANT: Now go to Runtime > Restart session,")
print("then run the cells below (skip this install cell).")
print("="*60)

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tqdm import tqdm
import pickle
import json
import zipfile
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## Step 2: Upload Kaggle API Credentials

**Instructions:**
1. Go to https://www.kaggle.com/account
2. Scroll to "API" section
3. Click "Create New API Token"
4. Upload the downloaded `kaggle.json` file below

In [ ]:
# Upload kaggle.json
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle credentials configured successfully!")

## Step 3: Download and Extract Datasets

In [ ]:
# Create directories
!mkdir -p datasets/raw
!mkdir -p datasets/processed
!mkdir -p models

# Download RAVDESS dataset (most popular)
print("Downloading RAVDESS dataset...")
!kaggle datasets download -d uwrfkaggle/ravdess-emotional-speech-audio -p datasets/raw --unzip

# Download CREMA-D dataset
print("\nDownloading CREMA-D dataset...")
!kaggle datasets download -d ejlok1/cremad -p datasets/raw --unzip

# Download TESS dataset
print("\nDownloading TESS dataset...")
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess -p datasets/raw --unzip

# Download SAVEE dataset (optional - smaller dataset)
print("\nDownloading SAVEE dataset...")
!kaggle datasets download -d ejlok1/surrey-audiovisual-expressed-emotion-savee -p datasets/raw --unzip

print("\n✓ All datasets downloaded successfully!")

## Step 4: Data Exploration

In [ ]:
# Function to extract emotion from RAVDESS filename
def extract_ravdess_emotion(filename):
    """
    RAVDESS filename format: 03-01-06-01-02-01-12.wav
    Third position (06) is emotion:
    01 = neutral, 02 = calm, 03 = happy, 04 = sad,
    05 = angry, 06 = fearful, 07 = disgust, 08 = surprised
    """
    emotion_map = {
        '01': 'neutral',
        '02': 'calm',
        '03': 'happy',
        '04': 'sad',
        '05': 'angry',
        '06': 'fear',
        '07': 'disgust',
        '08': 'surprised'
    }
    parts = filename.split('-')
    if len(parts) >= 3:
        return emotion_map.get(parts[2], 'unknown')
    return 'unknown'

# Function to extract emotion from CREMA-D filename
def extract_crema_emotion(filename):
    """
    CREMA-D filename format: 1001_DFA_ANG_XX.wav
    Third part is emotion: ANG, DIS, FEA, HAP, NEU, SAD
    """
    emotion_map = {
        'ANG': 'angry',
        'DIS': 'disgust',
        'FEA': 'fear',
        'HAP': 'happy',
        'NEU': 'neutral',
        'SAD': 'sad'
    }
    parts = filename.split('_')
    if len(parts) >= 3:
        return emotion_map.get(parts[2], 'unknown')
    return 'unknown'

# Function to extract emotion from TESS filename
def extract_tess_emotion(filename):
    """
    TESS filename format: YAF_dog_angry.wav or OAF_back_fear.wav
    Last part before .wav is emotion
    """
    emotion = filename.replace('.wav', '').split('_')[-1].lower()
    return emotion

# Function to extract emotion from SAVEE filename
def extract_savee_emotion(filename):
    """
    SAVEE filename format: DC_a01.wav, JE_h01.wav
    First letter after underscore is emotion:
    a = angry, d = disgust, f = fear, h = happy, n = neutral, sa = sad, su = surprised
    """
    emotion_map = {
        'a': 'angry',
        'd': 'disgust',
        'f': 'fear',
        'h': 'happy',
        'n': 'neutral',
        'sa': 'sad',
        'su': 'surprised'
    }
    parts = filename.replace('.wav', '').split('_')
    if len(parts) >= 2:
        code = parts[1][:2] if parts[1][:2] in ['sa', 'su'] else parts[1][0]
        return emotion_map.get(code, 'unknown')
    return 'unknown'

print("✓ Emotion extraction functions defined!")

In [ ]:
# Build dataset inventory
import glob

dataset_files = []

# Process RAVDESS
ravdess_path = 'datasets/raw/Actor_*'
for actor_folder in glob.glob(ravdess_path):
    if not os.path.isdir(actor_folder):
        continue
    for file in os.listdir(actor_folder):
        if file.endswith('.wav'):
            emotion = extract_ravdess_emotion(file)
            dataset_files.append({
                'path': os.path.join(actor_folder, file),
                'emotion': emotion,
                'dataset': 'RAVDESS'
            })

# Process CREMA-D
crema_path = 'datasets/raw/AudioWAV'
if os.path.exists(crema_path):
    for file in os.listdir(crema_path):
        if file.endswith('.wav'):
            emotion = extract_crema_emotion(file)
            dataset_files.append({
                'path': os.path.join(crema_path, file),
                'emotion': emotion,
                'dataset': 'CREMA-D'
            })

# Process TESS
tess_path = 'datasets/raw/TESS Toronto emotional speech set data'
if os.path.exists(tess_path):
    for subfolder in os.listdir(tess_path):
        subfolder_path = os.path.join(tess_path, subfolder)
        if os.path.isdir(subfolder_path):
            for file in os.listdir(subfolder_path):
                if file.endswith('.wav'):
                    emotion = extract_tess_emotion(file)
                    dataset_files.append({
                        'path': os.path.join(subfolder_path, file),
                        'emotion': emotion,
                        'dataset': 'TESS'
                    })

# Process SAVEE
savee_path = 'datasets/raw/ALL'
if os.path.exists(savee_path):
    for file in os.listdir(savee_path):
        if file.endswith('.wav'):
            emotion = extract_savee_emotion(file)
            dataset_files.append({
                'path': os.path.join(savee_path, file),
                'emotion': emotion,
                'dataset': 'SAVEE'
            })

# Create DataFrame
df = pd.DataFrame(dataset_files)

# Remove unknown emotions
df = df[df['emotion'] != 'unknown']

print(f"Total audio files: {len(df)}")
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())
print(f"\nDataset distribution:")
print(df['dataset'].value_counts())

# Visualize emotion distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df['emotion'].value_counts().plot(kind='bar')
plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
df['dataset'].value_counts().plot(kind='bar', color='orange')
plt.title('Dataset Distribution')
plt.xlabel('Dataset')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Step 5: Feature Extraction

In [ ]:
def extract_features(audio_path, duration=3.0, sr=22050, augment=None):
    """
    Extract audio features for emotion detection.

    Captures BOTH mean and std over time for every feature, preserving
    temporal variation and doubling the feature count from 190 -> 380.

    augment: None | 'noise' | 'pitch' | 'stretch'  (applied to the waveform)
    """
    try:
        y, sr = librosa.load(audio_path, duration=duration, sr=sr)

        target_length = int(duration * sr)
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        else:
            y = y[:target_length]

        # --- Optional waveform augmentation ---
        if augment == 'noise':
            y = y + 0.005 * np.random.randn(len(y))
        elif augment == 'pitch':
            y = librosa.effects.pitch_shift(y, sr=sr, n_steps=2)
        elif augment == 'stretch':
            y = librosa.effects.time_stretch(y, rate=0.9)
            if len(y) < target_length:
                y = np.pad(y, (0, target_length - len(y)), mode='constant')
            else:
                y = y[:target_length]

        def stats(feat):
            """mean + std across the time axis"""
            return np.concatenate([np.mean(feat.T, axis=0), np.std(feat.T, axis=0)])

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfcc_feat = stats(mfcc)

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_feat = stats(librosa.power_to_db(mel))

        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_feat = stats(chroma)

        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_feat = stats(contrast)

        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_feat = stats(zcr)

        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        rolloff_feat = stats(rolloff)

        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        centroid_feat = stats(centroid)

        features = np.hstack([
            mfcc_feat, mel_feat, chroma_feat,
            contrast_feat, zcr_feat, rolloff_feat, centroid_feat
        ])

        return features

    except Exception as e:
        print(f"Error processing {audio_path}: {str(e)}")
        return None

print("✓ Feature extraction function defined (mean + std)!")
print("Features: 2 x (MFCC40 + Mel128 + Chroma12 + Contrast7 + ZCR1 + Rolloff1 + Centroid1) = 380 features")

In [ ]:
# Test feature extraction on one file
test_file = df['path'].iloc[0]
test_features = extract_features(test_file)
print(f"Test file: {test_file}")
print(f"Feature shape: {test_features.shape}")
print(f"Feature sample (first 10): {test_features[:10]}")

## Step 6: Extract Features from All Audio Files

**Note**: This will take 15-30 minutes depending on dataset size. Processing ~12,000 audio files.

In [ ]:
# Extract features from all files
print("Extracting features from all audio files...")
print("This may take 15-30 minutes depending on dataset size.\n")

features_list = []
labels_list = []
failed_files = []

# Original + 3 augmented copies (noise, pitch, stretch) per file => ~4x data.
# To speed up, restrict augmentation to rare classes by editing AUGMENTATIONS.
AUGMENTATIONS = ['noise', 'pitch', 'stretch']

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing audio files"):
    features = extract_features(row['path'], augment=None)
    if features is not None:
        features_list.append(features)
        labels_list.append(row['emotion'])
    else:
        failed_files.append(row['path'])
        continue

    for aug in AUGMENTATIONS:
        features_aug = extract_features(row['path'], augment=aug)
        if features_aug is not None:
            features_list.append(features_aug)
            labels_list.append(row['emotion'])

print(f"\n✓ Feature extraction complete!")
print(f"Successfully processed: {len(features_list)} files")
print(f"Failed: {len(failed_files)} files")

# Convert to numpy arrays
X = np.array(features_list)
y = np.array(labels_list)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique emotions: {np.unique(y)}")

In [ ]:
# Save extracted features (backup)
np.save('datasets/processed/features.npy', X)
np.save('datasets/processed/labels.npy', y)
print("✓ Features and labels saved!")

## Step 7: Prepare Data for Training

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = keras.utils.to_categorical(y_encoded)

num_classes = len(label_encoder.classes_)
print(f"Number of emotion classes: {num_classes}")
print(f"Emotion mapping: {dict(enumerate(label_encoder.classes_))}")

# Save label encoder
with open('models/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("✓ Label encoder saved!")

In [ ]:
# Split data: 80% train, 10% validation, 10% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# Normalize features
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)

# Save normalization parameters
np.save('models/feature_mean.npy', mean)
np.save('models/feature_std.npy', std)
print("✓ Normalization parameters saved!")

## Step 8: Build CNN Model Architecture

In [ ]:
def create_emotion_model(input_shape, num_classes):
    """
    Create a deep neural network for speech emotion recognition

    Architecture:
    - Dense layers with batch normalization and dropout
    - Suitable for feature vector input
    """
    model = models.Sequential([
        # Input layer
        layers.Input(shape=(input_shape,)),

        # Dense Block 1
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Dense Block 2
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Dense Block 3
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Dense Block 4
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

# Create model
model = create_emotion_model(X_train.shape[1], num_classes)

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

# Print model summary
model.summary()

## Step 9: Train the Model

Training with:
- **Epochs**: 100 (with early stopping)
- **Batch Size**: 32
- **Optimizer**: Adam
- **Callbacks**: Early stopping, learning rate reduction, model checkpoint

In [ ]:
# Define callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(
    'models/best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Compute balanced class weights so under-represented emotions raise recall.
from sklearn.utils.class_weight import compute_class_weight

y_train_int = np.argmax(y_train, axis=1)
weights = compute_class_weight('balanced', classes=np.unique(y_train_int), y=y_train_int)
class_weight = dict(enumerate(weights))
print("Class weights:", class_weight)

# Train model
print("Starting training...\n")
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr, checkpoint],
    class_weight=class_weight,
    verbose=1
)

print("\n✓ Training complete!")

## Step 10: Visualize Training Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0, 0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0, 0].set_title('Model Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Loss
axes[0, 1].plot(history.history['loss'], label='Train Loss')
axes[0, 1].plot(history.history['val_loss'], label='Val Loss')
axes[0, 1].set_title('Model Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision
axes[1, 0].plot(history.history['precision'], label='Train Precision')
axes[1, 0].plot(history.history['val_precision'], label='Val Precision')
axes[1, 0].set_title('Model Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Recall
axes[1, 1].plot(history.history['recall'], label='Train Recall')
axes[1, 1].plot(history.history['val_recall'], label='Val Recall')
axes[1, 1].set_title('Model Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('models/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training plots saved!")

## Step 11: Evaluate Model on Test Set

In [ ]:
# Load best model
best_model = keras.models.load_model('models/best_model.h5')

# Evaluate on test set
test_loss, test_accuracy, test_precision, test_recall = best_model.evaluate(X_test, y_test, verbose=0)

print("=" * 50)
print("TEST SET EVALUATION RESULTS")
print("=" * 50)
print(f"Test Accuracy:  {test_accuracy*100:.2f}%")
print(f"Test Precision: {test_precision*100:.2f}%")
print(f"Test Recall:    {test_recall*100:.2f}%")
print(f"Test F1-Score:  {2 * (test_precision * test_recall) / (test_precision + test_recall)*100:.2f}%")
print(f"Test Loss:      {test_loss:.4f}")
print("=" * 50)

In [ ]:
# Generate predictions
y_pred = best_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Classification report
print("\nCLASSIFICATION REPORT:")
print("=" * 80)
print(classification_report(
    y_test_classes,
    y_pred_classes,
    target_names=label_encoder.classes_,
    digits=4
))

# Confusion Matrix
cm = confusion_matrix(y_test_classes, y_pred_classes)
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('Confusion Matrix - Speech Emotion Recognition', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Emotion', fontsize=12)
plt.ylabel('True Emotion', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved!")

## Step 12: Save Training Metrics and Configuration

In [ ]:
def extract_features(audio_path, duration=3.0, sr=22050, augment=None):
    """
    Extract audio features for emotion detection.
    Now captures BOTH mean and std over time (temporal variation),
    doubling the feature count from 190 -> 380.

    augment: None, 'noise', 'pitch', or 'stretch' (applied to the waveform)
    """
    try:
        y, sr = librosa.load(audio_path, duration=duration, sr=sr)

        # Pad or trim to exact duration
        target_length = int(duration * sr)
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        else:
            y = y[:target_length]

        # --- Optional augmentation (waveform level) ---
        if augment == 'noise':
            y = y + 0.005 * np.random.randn(len(y))
        elif augment == 'pitch':
            y = librosa.effects.pitch_shift(y, sr=sr, n_steps=2)
        elif augment == 'stretch':
            y = librosa.effects.time_stretch(y, rate=0.9)
            # re-fix length after stretching
            if len(y) < target_length:
                y = np.pad(y, (0, target_length - len(y)), mode='constant')
            else:
                y = y[:target_length]

        def stats(feat):
            """mean + std over time axis"""
            return np.concatenate([np.mean(feat.T, axis=0), np.std(feat.T, axis=0)])

        # 1. MFCC (40)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfcc_feat = stats(mfcc)

        # 2. Mel-Spectrogram (128)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_feat = stats(librosa.power_to_db(mel))

        # 3. Chroma (12)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_feat = stats(chroma)

        # 4. Spectral Contrast (7)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_feat = stats(contrast)

        # 5. Zero Crossing Rate (1)
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_feat = stats(zcr)

        # 6. Spectral Rolloff (1)
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        rolloff_feat = stats(rolloff)

        # 7. Spectral Centroid (1)
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        centroid_feat = stats(centroid)

        features = np.hstack([
            mfcc_feat, mel_feat, chroma_feat,
            contrast_feat, zcr_feat, rolloff_feat, centroid_feat
        ])

        return features

    except Exception as e:
        print(f"Error processing {audio_path}: {str(e)}")
        return None

print("✓ Feature extraction function defined (mean + std)!")
print("Features: 2 x (MFCC40 + Mel128 + Chroma12 + Contrast7 + ZCR1 + Rolloff1 + Centroid1) = 380 features")

## Step 13: Test Model with Sample Audio

In [ ]:
def predict_emotion(audio_path, model, label_encoder, mean, std):
    """
    Predict emotion from audio file
    """
    # Extract features
    features = extract_features(audio_path)
    if features is None:
        return None

    # Normalize
    features = (features - mean) / (std + 1e-8)

    # Reshape for model
    features = features.reshape(1, -1)

    # Predict
    prediction = model.predict(features, verbose=0)
    emotion_idx = np.argmax(prediction)
    emotion = label_encoder.classes_[emotion_idx]
    confidence = prediction[0][emotion_idx]

    # Get all probabilities
    all_probs = {label_encoder.classes_[i]: float(prediction[0][i]) for i in range(len(label_encoder.classes_))}

    return {
        'emotion': emotion,
        'confidence': float(confidence),
        'all_probabilities': all_probs
    }

# Test on random samples from test set
print("Testing on random samples from test set:\n")
print("=" * 80)

# Get actual file paths for test samples
test_indices = np.random.choice(len(df), 5, replace=False)

for idx in test_indices:
    sample_path = df.iloc[idx]['path']
    true_emotion = df.iloc[idx]['emotion']

    result = predict_emotion(sample_path, best_model, label_encoder, mean, std)

    if result:
        print(f"\nFile: {os.path.basename(sample_path)}")
        print(f"True Emotion:      {true_emotion}")
        print(f"Predicted Emotion: {result['emotion']}")
        print(f"Confidence:        {result['confidence']*100:.2f}%")
        print(f"Match: {'✓ CORRECT' if result['emotion'] == true_emotion else '✗ INCORRECT'}")
        print("-" * 80)

print("\n✓ Sample predictions complete!")

## Step 14: Package Model for Deployment

In [ ]:
# Create deployment package
print("Creating deployment package...\n")

# Save final model in multiple formats
best_model.save('models/speech_emotion_model.h5')
print("✓ Model saved as .h5 format")

# Save in SavedModel format (TensorFlow)
# Keras 3 (TF 2.16+) uses .export() for the SavedModel directory format
best_model.export('models/speech_emotion_model')
print("✓ Model saved in TensorFlow SavedModel format")

# Create README for the model
readme_content = f"""
# Speech Emotion Recognition Model

## Model Information
- **Architecture**: Deep Neural Network
- **Input**: Audio features (380-dimensional vector)
- **Output**: {num_classes} emotion classes
- **Accuracy**: {test_accuracy*100:.2f}%
- **F1-Score**: {metrics['test_f1']*100:.2f}%

## Emotion Classes
{', '.join(label_encoder.classes_)}

## Datasets Used
- RAVDESS (Ryerson Audio-Visual Database of Emotional Speech and Song)
- CREMA-D (Crowd-sourced Emotional Multimodal Actors Dataset)
- TESS (Toronto Emotional Speech Set)
- SAVEE (Surrey Audio-Visual Expressed Emotion)

Total training samples: {metrics['total_samples']}

## Features Extracted
- MFCC (40 coefficients)
- Mel-Spectrogram (128 bands)
- Chroma (12 features)
- Spectral Contrast (7 features)
- Zero Crossing Rate (1 feature)
- Spectral Rolloff (1 feature)
- Spectral Centroid (1 feature)

Total: 380 features (mean + std of each feature group)

## Usage

```python
import numpy as np
import librosa
import tensorflow as tf
import pickle

# Load model
model = tf.keras.models.load_model('speech_emotion_model.h5')

# Load label encoder and normalization params
with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

mean = np.load('feature_mean.npy')
std = np.load('feature_std.npy')

# Extract features from audio (use the extract_features function)
features = extract_features('path/to/audio.wav')

# Normalize
features = (features - mean) / (std + 1e-8)

# Predict
prediction = model.predict(features.reshape(1, -1))
emotion = label_encoder.classes_[np.argmax(prediction)]
confidence = prediction[0][np.argmax(prediction)]

print(f"Emotion: {{emotion}}, Confidence: {{confidence*100:.2f}}%")
```

## Files Included
- `speech_emotion_model.h5` - Trained model (Keras format)
- `speech_emotion_model/` - TensorFlow SavedModel format
- `label_encoder.pkl` - Label encoder for emotion classes
- `feature_mean.npy` - Feature normalization mean
- `feature_std.npy` - Feature normalization std
- `training_metrics.json` - Training metrics and configuration
- `confusion_matrix.png` - Confusion matrix visualization
- `training_history.png` - Training history plots

## Training Details
- Epochs: {metrics['training_epochs']}
- Batch Size: {metrics['batch_size']}
- Optimizer: {metrics['optimizer']}
- Learning Rate: {metrics['learning_rate']}

## Performance Metrics
- Accuracy: {test_accuracy*100:.2f}%
- Precision: {test_precision*100:.2f}%
- Recall: {test_recall*100:.2f}%
- F1-Score: {metrics['test_f1']*100:.2f}%

## Citation
If you use this model, please cite the datasets:
- RAVDESS: Livingstone SR, Russo FA (2018) The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS)
- CREMA-D: Cao H, Cooper DG, Keutmann MK, Gur RC, Nenkova A, Verma R (2014)
- TESS: Dupuis K, Pichora-Fuller MK (2010)
- SAVEE: Haq S, Jackson PJB (2011)
"""

with open('models/README.md', 'w') as f:
    f.write(readme_content)

print("✓ README created")

# Create a Python inference script
inference_script = '''
import numpy as np
import librosa
import tensorflow as tf
import pickle
import warnings
warnings.filterwarnings('ignore')

class SpeechEmotionRecognizer:
    def __init__(self, model_path, label_encoder_path, mean_path, std_path):
        """Initialize Speech Emotion Recognizer"""
        self.model = tf.keras.models.load_model(model_path)

        with open(label_encoder_path, 'rb') as f:
            self.label_encoder = pickle.load(f)

        self.mean = np.load(mean_path)
        self.std = np.load(std_path)

    def extract_features(self, audio_path, duration=3.0, sr=22050):
        """Extract audio features (mean + std, 380-dim to match training)"""
        try:
            y, sr = librosa.load(audio_path, duration=duration, sr=sr)

            target_length = int(duration * sr)
            if len(y) < target_length:
                y = np.pad(y, (0, target_length - len(y)), mode='constant')
            else:
                y = y[:target_length]

            def stats(feat):
                return np.concatenate([np.mean(feat.T, axis=0), np.std(feat.T, axis=0)])

            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
            mfcc_feat = stats(mfcc)

            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
            mel_feat = stats(librosa.power_to_db(mel))

            chroma = librosa.feature.chroma_stft(y=y, sr=sr)
            chroma_feat = stats(chroma)

            contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
            contrast_feat = stats(contrast)

            zcr = librosa.feature.zero_crossing_rate(y)
            zcr_feat = stats(zcr)

            rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
            rolloff_feat = stats(rolloff)

            centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
            centroid_feat = stats(centroid)

            features = np.hstack([
                mfcc_feat, mel_feat, chroma_feat,
                contrast_feat, zcr_feat, rolloff_feat, centroid_feat
            ])

            return features
        except Exception as e:
            print(f"Error extracting features: {e}")
            return None

    def predict(self, audio_path):
        """Predict emotion from audio file"""
        features = self.extract_features(audio_path)
        if features is None:
            return None

        features = (features - self.mean) / (self.std + 1e-8)
        features = features.reshape(1, -1)

        prediction = self.model.predict(features, verbose=0)
        emotion_idx = np.argmax(prediction)
        emotion = self.label_encoder.classes_[emotion_idx]
        confidence = prediction[0][emotion_idx]

        all_probs = {
            self.label_encoder.classes_[i]: float(prediction[0][i])
            for i in range(len(self.label_encoder.classes_))
        }

        return {
            'emotion': emotion,
            'confidence': float(confidence),
            'all_probabilities': all_probs
        }

# Example usage
if __name__ == "__main__":
    recognizer = SpeechEmotionRecognizer(
        model_path='speech_emotion_model.h5',
        label_encoder_path='label_encoder.pkl',
        mean_path='feature_mean.npy',
        std_path='feature_std.npy'
    )

    result = recognizer.predict('path/to/audio.wav')
    if result:
        print(f"Emotion: {result['emotion']}")
        print(f"Confidence: {result['confidence']*100:.2f}%")
        print(f"All probabilities: {result['all_probabilities']}")
'''

with open('models/inference.py', 'w') as f:
    f.write(inference_script)

print("✓ Inference script created")
print("\n" + "=" * 80)
print("MODEL PACKAGING COMPLETE!")
print("=" * 80)

## Step 15: Download Model Files

Download all model files to integrate into your backend.

In [ ]:
# Create a zip file with all model files
import shutil

print("Creating deployment package...\n")

# Create zip file
shutil.make_archive('speech_emotion_model_package', 'zip', 'models')

print("✓ Deployment package created: speech_emotion_model_package.zip")
print("\nPackage contains:")
print("  - speech_emotion_model.h5 (Keras model)")
print("  - speech_emotion_model/ (TensorFlow SavedModel)")
print("  - label_encoder.pkl")
print("  - feature_mean.npy")
print("  - feature_std.npy")
print("  - training_metrics.json")
print("  - confusion_matrix.png")
print("  - training_history.png")
print("  - README.md")
print("  - inference.py")

# Download the zip file
print("\nDownloading package...")
files.download('speech_emotion_model_package.zip')

print("\n" + "=" * 80)
print("🎉 TRAINING COMPLETE! 🎉")
print("=" * 80)
print(f"\nFinal Model Performance:")
print(f"  Accuracy:  {test_accuracy*100:.2f}%")
print(f"  Precision: {test_precision*100:.2f}%")
print(f"  Recall:    {test_recall*100:.2f}%")
print(f"  F1-Score:  {metrics['test_f1']*100:.2f}%")
print("\nNext Steps:")
print("1. Extract speech_emotion_model_package.zip")
print("2. Copy files to backend/app/ml/speech_emotion/trained_models/")
print("3. Update your inference code to use the new model")
print("4. Test with real audio samples")
print("=" * 80)